# Day 091 Solution — Strategy Comparison Dashboard

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def sma(s, w=20):  return s.rolling(window=w).mean()
def ema(s, w=20):  return s.ewm(span=w, adjust=False).mean()
def rsi(s, w=14):
    import warnings
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def macd(s, fast=12, slow=26, sig=9):
    ml = ema(s, fast) - ema(s, slow)
    sl = ema(ml, sig)
    return pd.DataFrame({"macd": ml, "signal": sl, "histogram": ml - sl})
def bollinger_bands(s, w=20, ns=2.0):
    mid = sma(s, w); std = s.rolling(w).std()
    return pd.DataFrame({"upper": mid+ns*std, "middle": mid, "lower": mid-ns*std})
def add_indicators(df, sw=20, ew=20, rw=14, mf=12, ms=26, mg=9, bw=20, bs=2.0):
    df = df.copy(); c = df["Close"]
    df[f"sma_{sw}"] = sma(c, sw); df[f"ema_{ew}"] = ema(c, ew)
    df[f"rsi_{rw}"] = rsi(c, rw)
    m = macd(c, mf, ms, mg)
    df["macd"] = m["macd"]; df["macd_signal"] = m["signal"]; df["macd_hist"] = m["histogram"]
    bb = bollinger_bands(c, bw, bs)
    df["bb_upper"] = bb["upper"]; df["bb_middle"] = bb["middle"]; df["bb_lower"] = bb["lower"]
    return df
def compute_returns(df):
    return df["Close"].pct_change()

def compute_equity(returns, initial=1.0):
    return (1 + returns.fillna(0)).cumprod() * initial
def max_drawdown(equity):
    peak = equity.cummax()
    return float(((equity - peak) / peak).min())
def sharpe_ratio(returns, periods_per_year=252):
    clean = returns.dropna()
    if len(clean) == 0 or clean.std() == 0:
        return 0.0
    return float(clean.mean() / clean.std() * (periods_per_year ** 0.5))
def run_backtest(df, signals):
    market_returns   = compute_returns(df)
    positions        = signals.shift(1).fillna(0)
    strategy_returns = positions * market_returns
    equity  = compute_equity(strategy_returns)
    clean   = strategy_returns.dropna()
    n_days  = len(clean)
    win_rate = float((clean > 0).sum() / max(n_days, 1))
    pos_diff = positions.diff().fillna(0)
    n_trades = int((pos_diff != 0).sum())
    total_ret = float(equity.iloc[-1] - 1.0)
    base = 1.0 + total_ret
    ann_ret = float(base ** (252.0 / max(n_days, 1)) - 1) if base > 0 else -1.0
    return {
        "total_return":      total_ret,
        "annualized_return": ann_ret,
        "sharpe_ratio":      sharpe_ratio(strategy_returns),
        "max_drawdown":      max_drawdown(equity),
        "win_rate":          win_rate,
        "n_trades":          n_trades,
        "equity":            equity,
        "strategy_returns":  strategy_returns,
        "market_returns":    market_returns,
    }


In [ ]:
df       = _synthetic(n=252)
enriched = add_indicators(df)

sig_bah = pd.Series(1, index=df.index)
sig_sma = (enriched["Close"] > enriched["sma_20"]).astype(int)
sig_rsi = pd.Series(float("nan"), index=enriched.index)
sig_rsi[enriched["rsi_14"] < 40]  = 1.0
sig_rsi[enriched["rsi_14"] > 60]  = 0.0
sig_rsi = sig_rsi.ffill().fillna(0.0)

r_bah = run_backtest(df, sig_bah)
r_sma = run_backtest(df, sig_sma)
r_rsi = run_backtest(df, sig_rsi)

# Correctness assertions
for r in [r_bah, r_sma, r_rsi]:
    assert isinstance(r["equity"], pd.Series) and len(r["equity"]) == len(df)
    assert abs(r["equity"].iloc[-1] - (1 + r["total_return"])) < 1e-9
    assert r["max_drawdown"] <= 1e-9
    assert 0.0 <= r["win_rate"] <= 1.0
    assert r["n_trades"] >= 0

# look-ahead: always-long positions[0] = 0
positions_bah = sig_bah.shift(1).fillna(0)
assert positions_bah.iloc[0] == 0, "look-ahead: position[0] should be 0"

print(f"{'Metric':<20} {'Buy-Hold':>10} {'SMA-20':>10} {'RSI-MR':>10}")
print("-" * 52)
for key, fmt in [
    ("total_return",      ".2%"),
    ("annualized_return", ".2%"),
    ("sharpe_ratio",      ".3f"),
    ("max_drawdown",      ".2%"),
    ("win_rate",          ".2%"),
    ("n_trades",          "d"),
]:
    v1 = r_bah[key]; v2 = r_sma[key]; v3 = r_rsi[key]
    if fmt == "d":
        print(f"{key:<20} {v1:>10d} {v2:>10d} {v3:>10d}")
    else:
        print(f"{key:<20} {v1:>{10}{fmt}} {v2:>{10}{fmt}} {v3:>{10}{fmt}}")
print("\nSolution smoke-test passed.")
